###  Import Libraray

In [44]:
import pandas as pd
import numpy as np

### Load the Raw Dataset

In [45]:
df = pd.read_csv('../data/row/credit_risk_dataset.csv')
print('Raw Data Shape:', df.shape)
df.head()

Raw Data Shape: (32581, 12)


,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4


### Rename Columns to Simple Names

In [46]:
column_mapping = {
    'person_age': 'age',
    'person_income': 'income',
    'person_home_ownership': 'home_ownership',
    'person_emp_length': 'employment_years',
    'loan_intent': 'loan_purpose',
    'loan_grade': 'loan_grade',
    'loan_amnt': 'loan_amount',
    'loan_int_rate': 'interest_rate',
    'loan_status': 'default',
    'loan_percent_income': 'loan_income_ratio',
    'cb_person_default_on_file': 'previous_default',
    'cb_person_cred_hist_length': 'credit_history_years'
}

df = df.rename(columns=column_mapping)
df.head()

,age,income,home_ownership,employment_years,loan_purpose,loan_grade,loan_amount,interest_rate,default,loan_income_ratio,previous_default,credit_history_years
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4


###  Check Column Types and Information

In [47]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32581 entries, 0 to 32580
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   age                   32581 non-null  int64  
 1   income                32581 non-null  int64  
 2   home_ownership        32581 non-null  str    
 3   employment_years      31686 non-null  float64
 4   loan_purpose          32581 non-null  str    
 5   loan_grade            32581 non-null  str    
 6   loan_amount           32581 non-null  int64  
 7   interest_rate         29465 non-null  float64
 8   default               32581 non-null  int64  
 9   loan_income_ratio     32581 non-null  float64
 10  previous_default      32581 non-null  str    
 11  credit_history_years  32581 non-null  int64  
dtypes: float64(3), int64(5), str(4)
memory usage: 3.5 MB


###  Check and Remove Duplicate Rows

In [48]:
print('Duplicates found:', df.duplicated().sum())
df = df.drop_duplicates()
print('Shape after removing duplicates:', df.shape)

Duplicates found: 165
Shape after removing duplicates: (32416, 12)


###  Check for Missing Values (Count)

In [49]:
print(df.isnull().sum())

age                        0
income                     0
home_ownership             0
employment_years         887
loan_purpose               0
loan_grade                 0
loan_amount                0
interest_rate           3095
default                    0
loan_income_ratio          0
previous_default           0
credit_history_years       0
dtype: int64


### Check Missing Values Percentage

In [50]:
missing_pct = (df.isnull().sum() / len(df)) * 100
print(missing_pct[missing_pct > 0].round(2))

employment_years    2.74
interest_rate       9.55
dtype: float64


###  Inpute Missing Interest Rate by Loan Grade Median

In [51]:
df['interest_rate'] = df['interest_rate'].fillna(
    df.groupby('loan_grade')['interest_rate'].transform('median')
)
print('Missing interest_rate remaining:', df['interest_rate'].isnull().sum())

Missing interest_rate remaining: 0


###  Impute Missing Employment Years with Median

In [52]:
df['employment_years'] = df['employment_years'].fillna(df['employment_years'].median())
print('Missing employment_years remaining:', df['employment_years'].isnull().sum())

Missing employment_years remaining: 0


###  Verify All Missing Values Are Filled

In [53]:
print('Total missing values in dataset:', df.isnull().sum().sum())

Total missing values in dataset: 0


###  Check and Filter Age Outliers (18 to 100 Years)

In [54]:
print('Records with age > 100:')
print(df[df['age'] > 100][['age', 'income', 'loan_amount']])

df = df[(df['age'] >= 18) & (df['age'] <= 100)]
print('Age range after cleaning:', df['age'].min(), 'to', df['age'].max())

Records with age > 100:
       age   income  loan_amount
81     144   250000         4800
183    144   200000         6000
575    123    80004        20400
747    123    78000        20000
32297  144  6000000         5000
Age range after cleaning: 20 to 94


###  Check and Filter Employment Years Outliers (> 60 Years)

In [55]:
print('Records with employment_years > 60:')
print(df[df['employment_years'] > 60][['age', 'income', 'employment_years']])

df = df[df['employment_years'] <= 60]
print('Max employment years after cleaning:', df['employment_years'].max())

Records with employment_years > 60:
     age  income  employment_years
0     22   59000             123.0
210   21  192000             123.0
Max employment years after cleaning: 41.0


###  Check That Employment Years Does Not Exceed Working Age

In [56]:
invalid_emp = df[df['employment_years'] > (df['age'] - 14)]
print('Records where employment_years > (age - 14):', len(invalid_emp))

df = df[df['employment_years'] <= (df['age'] - 14)]
print('Shape after employment validation:', df.shape)

Records where employment_years > (age - 14): 0
Shape after employment validation: (32409, 12)


###  Verify Income Values (No Negative or Zero Values)

In [57]:
print('Records with income <= 0:', (df['income'] <= 0).sum())
print('Min income:', df['income'].min())
print('Max income:', df['income'].max())

Records with income <= 0: 0
Min income: 4000
Max income: 2039784


###  Verify Loan Amount Values

In [58]:
print('Records with loan_amount <= 0:', (df['loan_amount'] <= 0).sum())
print('Min loan amount:', df['loan_amount'].min())
print('Max loan amount:', df['loan_amount'].max())

Records with loan_amount <= 0: 0
Min loan amount: 500
Max loan amount: 35000


###  Verify Loan-to-Income Ratio Range

In [59]:
print('Records with loan_income_ratio > 1.0:', (df['loan_income_ratio'] > 1.0).sum())
print('Records with loan_income_ratio < 0:', (df['loan_income_ratio'] < 0).sum())
print('Min ratio:', df['loan_income_ratio'].min())
print('Max ratio:', df['loan_income_ratio'].max())

Records with loan_income_ratio > 1.0: 0
Records with loan_income_ratio < 0: 0
Min ratio: 0.0
Max ratio: 0.83


###  Standardize Categorical Strings (Trim Whitespace)

In [60]:
cat_cols = ['home_ownership', 'loan_purpose', 'loan_grade', 'previous_default']
for col in cat_cols:
    df[col] = df[col].astype(str).str.strip().str.upper()

print('Categorical columns standardized successfully.')

Categorical columns standardized successfully.


###  Check Unique Categories in Categorical Features

In [61]:
print('Home Ownership:', df['home_ownership'].unique())
print('Loan Purpose:', df['loan_purpose'].unique())
print('Loan Grade:', sorted(df['loan_grade'].unique()))
print('Previous Default:', df['previous_default'].unique())

Home Ownership: <ArrowStringArray>
['OWN', 'MORTGAGE', 'RENT', 'OTHER']
Length: 4, dtype: str
Loan Purpose: <ArrowStringArray>
[        'EDUCATION',           'MEDICAL',           'VENTURE',
          'PERSONAL',   'HOMEIMPROVEMENT', 'DEBTCONSOLIDATION']
Length: 6, dtype: str
Loan Grade: ['A', 'B', 'C', 'D', 'E', 'F', 'G']
Previous Default: <ArrowStringArray>
['N', 'Y']
Length: 2, dtype: str


###  Check Target Variable (Default) Counts and Proportions

In [62]:
print('Default Counts:')
print(df['default'].value_counts())
print('\nDefault Percentage:')
print((df['default'].value_counts(normalize=True) * 100).round(2))

Default Counts:
default
0    25321
1     7088
Name: count, dtype: int64

Default Percentage:
default
0    78.13
1    21.87
Name: proportion, dtype: float64


### Final Overview of Cleaned Data

In [63]:
print('Cleaned Data Shape:', df.shape)
print('Null values count:', df.isnull().sum().sum())
print('Duplicates count:', df.duplicated().sum())
df.describe().round(2)

Cleaned Data Shape: (32409, 12)
Null values count: 0
Duplicates count: 0


,age,income,employment_years,loan_amount,interest_rate,default,loan_income_ratio,credit_history_years
count,32409.00,32409.00,32409.00,32409.00,32409.00,32409.00,32409.00,32409.00
mean,27.73,65894.28,4.76,9592.49,11.02,0.22,0.17,5.81
std,6.21,52517.87,3.98,6320.89,3.21,0.41,0.11,4.06
min,20.00,4000.00,0.00,500.00,5.42,0.00,0.00,2.00
25%,23.00,38500.00,2.00,5000.00,7.88,0.00,0.09,3.00
50%,26.00,55000.00,4.00,8000.00,10.99,0.00,0.15,4.00
75%,30.00,79200.00,7.00,12250.00,13.48,0.00,0.23,8.00
max,94.00,2039784.00,41.00,35000.00,23.22,1.00,0.83,30.00


###  Save Cleaned Dataset to CSV

In [64]:
output_path = '../data/processed/cleaned_data.csv'
df.to_csv(output_path, index=False)
print(f'Cleaned dataset successfully saved to: {output_path}')

Cleaned dataset successfully saved to: ../data/processed/cleaned_data.csv
